# Unidad de análisis: ¿con qué podemos perfilar?

Notebook exploratorio. El objetivo del proyecto es **perfilar y recomendar**; acá se valida qué entidad se puede perfilar con los datos disponibles:

1. **Review** (`user_id`, `parent_asin`, `timestamp`) enriquecida con la metadata del producto: la tabla base.
2. **Usuario**: ¿hay historial suficiente por usuario para armar un perfil? → prueba de agrupar por `user_id`, en la muestra y en una categoría completa.
3. **Producto**: ¿qué tan completa es la metadata para perfilar ítems por contenido?

Consume `data/samples/` (generado por [02_stratified_sampling.ipynb](02_stratified_sampling.ipynb)). Solo local.

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

from proyecto_final_ceia.sampling import REVIEWS_PATH_TMPL, stream_jsonl

SAMPLES_DIR = Path("../data/samples")
ANALYSIS_DIR = Path("../data/analysis")
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

reviews_df = pd.read_parquet(SAMPLES_DIR / "reviews_sample.parquet")
meta_df = pd.read_parquet(SAMPLES_DIR / "meta_sample.parquet")
print(f"reviews: {reviews_df.shape}, metadata: {meta_df.shape}")

reviews: (30000, 11), metadata: (20959, 15)


## 1. Tabla base: review + metadata del producto

Left join por `parent_asin`. Las columnas no hasheables (listas/dicts) se achatan a texto o conteos para poder agrupar y describir sin colgar pandas (ver `agents/WORKFLOW.md`).

In [2]:
def as_list(x) -> list:
    """Parquet devuelve las listas como np.ndarray; normaliza a list (None -> [])."""
    if isinstance(x, np.ndarray):
        return x.tolist()
    if isinstance(x, list):
        return x
    return []


def flatten_meta(meta: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame({
        "parent_asin": meta["parent_asin"],
        "main_category": meta["main_category"],
        "product_title": meta["title"],
        "average_rating": meta["average_rating"],
        "rating_number": meta["rating_number"],
        "price": meta["price"],
        "store": meta["store"],
    })
    out["brand"] = meta["details"].map(lambda d: d.get("Brand") if isinstance(d, dict) else None)
    out["features_text"] = meta["features"].map(lambda x: " ".join(as_list(x)))
    out["description_text"] = meta["description"].map(lambda x: " ".join(as_list(x)))
    out["categories_path"] = meta["categories"].map(lambda x: " > ".join(as_list(x)))
    out["n_product_images"] = meta["images"].map(lambda x: len(as_list(x)))
    out["n_videos"] = meta["videos"].map(lambda x: len(as_list(x)))
    out["n_bought_together"] = meta["bought_together"].map(lambda x: len(as_list(x)))
    return out

In [3]:
reviews_flat = reviews_df.rename(columns={"title": "review_title", "text": "review_text"}).drop(columns=["images"])
reviews_flat["n_review_images"] = reviews_df["images"].map(lambda x: len(as_list(x)))

enriched = reviews_flat.merge(flatten_meta(meta_df), on="parent_asin", how="left", validate="many_to_one")
enriched["date"] = pd.to_datetime(enriched["timestamp"], unit="ms")
enriched["review_len"] = enriched["review_text"].str.len()
enriched["is_positive"] = enriched["rating"] >= 4

assert len(enriched) == len(reviews_df), "el join duplicó o perdió filas"
print(f"filas: {len(enriched):,} | sin metadata: {enriched['product_title'].isna().sum():,}")
enriched.head(3).T

filas: 30,000 | sin metadata: 0


,0,1,2
rating,1.0,1.0,1.0
review_title,Terrible lashes,Wouldn’t dry or go on even.,Doesn’t work!
review_text,They are way different then the picture they a...,This was awful,It worked for about a week and then would no l...
asin,B07MSDRY4K,B07N37TMK7,B07XY659KF
parent_asin,B07MSDRY4K,B07N37TMK7,B0C17MLTNF
user_id,AFQFCIAPBWCC76STIBEXVEQGGWAQ,AE7GWJAPV4ULU6XVJJAYDBRJTFGQ,AF4Q2VXNN6CED7G2EZG2GVLPV2WQ
timestamp,1556571815191,1562794952019,1614474408482
helpful_vote,6,0,0
verified_purchase,True,True,True
category,All_Beauty,All_Beauty,All_Beauty


## 2. Prueba: agrupar por usuario

Para perfilar usuarios hace falta historial: varias reviews por `user_id`. Primero en la muestra.

In [4]:
def history_distribution(counts: pd.Series) -> pd.DataFrame:
    """Resume una serie de #reviews por usuario (o por producto) en umbrales útiles."""
    thresholds = [1, 2, 3, 5, 10, 20]
    return pd.DataFrame({
        "entidades_con_>=n": [(counts >= n).sum() for n in thresholds],
        "%_entidades": [round(100 * (counts >= n).mean(), 2) for n in thresholds],
        "%_reviews_cubiertas": [round(100 * counts[counts >= n].sum() / counts.sum(), 2) for n in thresholds],
    }, index=pd.Index(thresholds, name="n"))


user_profile_sample = enriched.groupby("user_id").agg(
    n_reviews=("rating", "size"),
    n_categories=("category", "nunique"),
    mean_rating=("rating", "mean"),
    mean_price=("price", "mean"),
    pct_verified=("verified_purchase", "mean"),
    mean_review_len=("review_len", "mean"),
)
print(f"usuarios en la muestra: {len(user_profile_sample):,}")
history_distribution(user_profile_sample["n_reviews"])

usuarios en la muestra: 29,831


,entidades_con_>=n,%_entidades,%_reviews_cubiertas
n,,,
1,29831,100.00,100.00
2,150,0.50,1.06
3,15,0.05,0.16
5,1,0.00,0.02
10,0,0.00,0.00
20,0,0.00,0.00


In [5]:
user_profile_sample.sort_values("n_reviews", ascending=False).head(10)

,n_reviews,n_categories,mean_rating,mean_price,pct_verified,mean_review_len
user_id,,,,,,
AG4D44BNNLUEZNG7COK2CNRYUHYQ,5,2,3.600000,NaN,0.000000,393.200000
AFDYIK3FNPY2JFBQYUWC6GSBMIRQ_1,4,2,3.250000,NaN,0.000000,295.750000
AFXF3EGQTQDXMRLDWFU7UBFQZB7Q,4,2,3.000000,NaN,0.000000,923.500000
AF2BLE54TEMGZ546U763ZHZRXC4A,3,1,3.666667,NaN,0.000000,2711.000000
AF4P6A3SEKS2HCISWAKFZVIFM7NQ,3,2,3.333333,NaN,0.000000,495.000000
AFXMJCVLRCGRAEC7P37FZHEB24UQ,3,1,2.666667,0.33,1.000000,77.000000
AEZY3WQ5CO4DOXRNJOWWXJYKCAXQ,3,2,3.666667,0.00,0.333333,292.333333
AEFRTLVCVRALKXBED77KHPIXEPWQ,3,1,3.666667,NaN,0.000000,351.666667
AFKZESU3PTCQ2UVDBDSFVFTXBFNQ_1,3,2,2.000000,NaN,0.000000,331.333333


### 2b. ¿Es un artefacto del muestreo?

El reservoir por `(categoría, rating)` toma reviews al azar, así que casi nunca agarra dos del mismo usuario aunque el usuario tenga muchas. Para separar "el dataset no tiene historial" de "la muestra lo rompió", se recorre **una categoría completa** (`All_Beauty`, la más chica: ~700k reviews) en streaming, contando solo `user_id` y `parent_asin` — sin guardar el raw.

Tarda unos minutos (descarga ~300MB por HTTP). Poner `RUN_FULL_CATEGORY = False` para saltearlo.

In [6]:
RUN_FULL_CATEGORY = True
FULL_CATEGORY = "All_Beauty"

if RUN_FULL_CATEGORY:
    user_counts, item_counts = Counter(), Counter()
    for row in stream_jsonl(REVIEWS_PATH_TMPL.format(category=FULL_CATEGORY)):
        user_counts[row["user_id"]] += 1
        item_counts[row["parent_asin"]] += 1
    full_user_counts = pd.Series(user_counts, name="n_reviews")
    full_item_counts = pd.Series(item_counts, name="n_reviews")
    print(f"{FULL_CATEGORY}: {full_user_counts.sum():,} reviews, {len(full_user_counts):,} usuarios, {len(full_item_counts):,} productos")

All_Beauty: 701,528 reviews, 631,986 usuarios, 112,565 productos


In [7]:
print("Reviews por usuario (categoría completa):")
display(history_distribution(full_user_counts))
print("Reviews por producto (categoría completa):")
display(history_distribution(full_item_counts))

Reviews por usuario (categoría completa):


,entidades_con_>=n,%_entidades,%_reviews_cubiertas
n,,,
1,631986,100.00,100.00
2,48433,7.66,16.82
3,9159,1.45,5.62
5,1620,0.26,2.14
10,330,0.05,1.02
20,117,0.02,0.62


Reviews por producto (categoría completa):


,entidades_con_>=n,%_entidades,%_reviews_cubiertas
n,,,
1,112565,100.00,100.00
2,64999,57.74,93.22
3,45125,40.09,87.55
5,27533,24.46,79.07
10,13257,11.78,65.89
20,6044,5.37,52.13


In [8]:
# Los usuarios que cayeron en la muestra de esta categoría: ¿cuánto historial tienen en realidad?
sampled_users = enriched.loc[enriched["category"] == FULL_CATEGORY, "user_id"].unique()
real_history = full_user_counts.reindex(sampled_users)
print(f"usuarios muestreados de {FULL_CATEGORY}: {len(sampled_users):,}")
history_distribution(real_history)

usuarios muestreados de All_Beauty: 9,960


,entidades_con_>=n,%_entidades,%_reviews_cubiertas
n,,,
1,9960,100.00,100.00
2,1590,15.96,43.87
3,543,5.45,29.83
5,196,1.97,22.11
10,83,0.83,17.28
20,44,0.44,13.96


## 3. Perfil de producto: cobertura de la metadata

Para perfilar ítems por contenido importa qué campos vienen completos, por categoría.

In [9]:
products = flatten_meta(meta_df).assign(category=meta_df["category"])

presence = pd.DataFrame({
    "title": products["product_title"].fillna("").str.len() > 0,
    "price": products["price"].notna(),
    "store": products["store"].fillna("").str.len() > 0,
    "brand": products["brand"].notna(),
    "features": products["features_text"].str.len() > 0,
    "description": products["description_text"].str.len() > 0,
    "categories": products["categories_path"].str.len() > 0,
    "images": products["n_product_images"] > 0,
    "bought_together": products["n_bought_together"] > 0,
    "category": products["category"],
})
(presence.groupby("category").mean() * 100).round(1).T

category,All_Beauty,Amazon_Fashion,Software
title,100.0,100.0,100.0
price,20.1,12.7,82.5
store,90.4,96.5,99.9
brand,66.3,7.0,0.0
features,22.8,61.5,98.4
description,20.3,10.0,97.9
categories,0.0,0.0,17.4
images,100.0,100.0,100.0
bought_together,0.0,0.0,0.0


In [10]:
# Texto disponible para un perfil de contenido (TF-IDF / embeddings)
products["content_len"] = (products["product_title"].fillna("") + " " + products["features_text"] + " " + products["description_text"]).str.len()
products.groupby("category")["content_len"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
category,,,,,,,,
All_Beauty,7476.0,380.0,626.0,2.0,101.0,155.0,203.0,12630.0
Amazon_Fashion,9300.0,264.0,459.0,9.0,91.0,113.0,155.0,7047.0
Software,4183.0,2001.0,3332.0,18.0,762.0,1319.0,2241.0,126363.0


## 4. Señales de la review (sirven para cualquier perfil)

In [11]:
enriched.groupby("category").agg(
    reviews=("rating", "size"),
    median_review_len=("review_len", "median"),
    pct_with_text=("review_len", lambda s: round(100 * (s > 0).mean(), 1)),
    pct_verified=("verified_purchase", lambda s: round(100 * s.mean(), 1)),
    pct_helpful=("helpful_vote", lambda s: round(100 * (s > 0).mean(), 1)),
    pct_with_images=("n_review_images", lambda s: round(100 * (s > 0).mean(), 1)),
    date_min=("date", "min"),
    date_max=("date", "max"),
)

,reviews,median_review_len,pct_with_text,pct_verified,pct_helpful,pct_with_images,date_min,date_max
category,,,,,,,,
All_Beauty,10000,114.0,100.0,90.4,27.8,8.0,2003-05-17 04:48:39,2023-08-04 00:47:32.251
Amazon_Fashion,10000,102.0,100.0,93.0,20.2,6.4,2005-08-01 21:21:41,2023-08-28 13:19:42.420
Software,10000,101.0,100.0,94.2,44.6,0.2,1999-12-01 18:32:46,2023-07-11 19:49:01.268


In [12]:
out_path = ANALYSIS_DIR / "reviews_enriched.parquet"
enriched.to_parquet(out_path, index=False)
print(f"-> {out_path} ({len(enriched):,} filas, {enriched.shape[1]} columnas)")

-> ..\data\analysis\reviews_enriched.parquet (30,000 filas, 27 columnas)


## Conclusiones

- **Agrupar por usuario en la muestra no sirve**: solo el 0,5% de los usuarios tiene ≥2 reviews. Es un artefacto del reservoir por `(categoría, rating)`.
- **Pero en la categoría completa el historial también es escaso**: en `All_Beauty`, el 7,7% de los usuarios tiene ≥2 reviews (16,8% de las reviews) y solo el 0,26% tiene ≥5. Un perfil de usuario *dentro de una categoría* sirve para una minoría.
- **El producto es la entidad perfilable**: el 58% de los productos tiene ≥2 reviews y esos cubren el 93% de las reviews; con ≥10 reviews hay un 12% de productos que cubre el 66%.
- **La calidad de la metadata depende de la categoría**: Software es rica en texto (features/description ~98%); Beauty y Fashion casi solo tienen título y store (brand 66% en Beauty, 7% en Fashion; price 13-20%).
- **`bought_together` está vacío (0%)**: la co-compra (HU4) no puede salir de la metadata; habría que derivarla de co-reviews del mismo usuario, lo que choca con el historial escaso.
- **Pendiente**: el `user_id` es global entre categorías. Falta medir el historial *cross-categoría*, porque un usuario con 1 review en Beauty puede tener varias en otras categorías.